# 05 — SPARQL queries


In [1]:
# ============================================================
# 05 — SPARQL KNOWLEDGE GRAPH QUERIES
# Smart City Knowledge Graph & Network Accessibility Analyzer
# ============================================================

from pathlib import Path
import sys

import pandas as pd
from rdflib import Graph, Namespace, RDF, RDFS, OWL

# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(
        f"Project root could not be identified: {PROJECT_ROOT}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("=" * 60)
print("SPARQL KNOWLEDGE GRAPH ANALYSIS")
print("=" * 60)

print(f"Project root: {PROJECT_ROOT}")


SPARQL KNOWLEDGE GRAPH ANALYSIS
Project root: /Users/subhankarbiswas/smart-city-knowledge-graph-v3


In [2]:
# ------------------------------------------------------------
# Load RDF/Turtle graph
# ------------------------------------------------------------

GRAPH_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "graphs"
    / "smart_city_kg.ttl"
)

if not GRAPH_PATH.exists():
    raise FileNotFoundError(
        f"Knowledge graph not found:\n{GRAPH_PATH}\n\n"
        "Run Notebook 04 first."
    )

g = Graph()

g.parse(
    GRAPH_PATH,
    format="turtle",
)

print("\n✓ Knowledge graph loaded")
print(f"File: {GRAPH_PATH}")
print(f"Triples: {len(g):,}")


✓ Knowledge graph loaded
File: /Users/subhankarbiswas/smart-city-knowledge-graph-v3/outputs/graphs/smart_city_kg.ttl
Triples: 49,930


In [3]:
# ------------------------------------------------------------
# RDF namespaces
# ------------------------------------------------------------

SC = Namespace(
    "https://example.org/smartcity/"
)

OWL_NS = OWL
RDFS_NS = RDFS

print("Smart City namespace:")
print(SC)

Smart City namespace:
https://example.org/smartcity/


In [4]:
# ------------------------------------------------------------
# Reusable SPARQL query function
# ------------------------------------------------------------

def run_query(name, query):
    """
    Execute a SPARQL query and return a DataFrame.
    """

    print("\n" + "=" * 60)
    print(name.upper())
    print("=" * 60)

    try:

        results = g.query(query)

        rows = [
            tuple(row)
            for row in results
        ]

        columns = [
            str(variable)
            for variable in results.vars
        ]

        df = pd.DataFrame(
            rows,
            columns=columns,
        )

        print(f"Rows returned: {len(df):,}")

        if not df.empty:
            display(df)

        else:
            print("No results found.")

        return df

    except Exception as exc:

        print(
            f"SPARQL query failed:\n{exc}"
        )

        return pd.DataFrame()

### Query 1 — Services / POIs ###

In [5]:
service_query = f"""
PREFIX sc: <{SC}>
PREFIX rdfs: <{RDFS_NS}>

SELECT ?x ?service ?label
WHERE {{
    ?x sc:serviceType ?service ;
       rdfs:label ?label .
}}
ORDER BY ?service ?label
LIMIT 100
"""

services_df = run_query(
    "Services / POIs",
    service_query,
)


SERVICES / POIS
Rows returned: 100


,x,service,label
0,https://example.org/entity/poi_7884771421,atm,Deutsche Bank
1,https://example.org/entity/poi_13912361077,atm,Sparkasse Werra-Meißner
2,https://example.org/entity/poi_11236817637,atm,Unnamed
3,https://example.org/entity/poi_11547191016,atm,Unnamed
4,https://example.org/entity/poi_11547191017,atm,Unnamed
...,...,...,...
95,https://example.org/entity/poi_4229685355,bus_stop,Höhenweg
96,https://example.org/entity/poi_4075103494,bus_stop,Im Berke
97,https://example.org/entity/poi_2093267536,bus_stop,Industriehof
98,https://example.org/entity/poi_2093267544,bus_stop,Industriehof


### Query 2 — Wikidata reconciliation ###

In [6]:
wikidata_query = f"""
PREFIX owl: <{OWL_NS}>

SELECT ?x ?wd
WHERE {{
    ?x owl:sameAs ?wd .
}}
ORDER BY ?x
LIMIT 100
"""

wikidata_df = run_query(
    "OSM ↔ Wikidata reconciliation",
    wikidata_query,
)


OSM ↔ WIKIDATA RECONCILIATION
Rows returned: 5


,x,wd
0,https://example.org/entity/poi_12110384939,https://www.wikidata.org/entity/Q28682476
1,https://example.org/entity/poi_179450968,https://www.wikidata.org/entity/Q116511109
2,https://example.org/entity/poi_2315563270,https://www.wikidata.org/entity/Q800740
3,https://example.org/entity/poi_280734435,https://www.wikidata.org/entity/Q116511061
4,https://example.org/entity/poi_44806741,https://www.wikidata.org/entity/Q44549210


### Query 3 — Buildings ###

In [7]:
building_query = f"""
PREFIX sc: <{SC}>
PREFIX rdf: <{RDF}>

SELECT ?x
WHERE {{
    ?x rdf:type sc:Building .
}}
LIMIT 100
"""

buildings_df = run_query(
    "Buildings",
    building_query,
)


BUILDINGS
Rows returned: 100


,x
0,https://example.org/entity/building_1009700303
1,https://example.org/entity/building_1009700304
2,https://example.org/entity/building_1009700305
3,https://example.org/entity/building_1009700306
4,https://example.org/entity/building_1009700307
...,...
95,https://example.org/entity/building_1020610183
96,https://example.org/entity/building_1020610184
97,https://example.org/entity/building_1020610185
98,https://example.org/entity/building_1020610186


### Query 4 — Count entities by class ###

In [8]:
entity_count_query = f"""
PREFIX rdf: <{RDF}>

SELECT ?type (COUNT(?x) AS ?count)
WHERE {{
    ?x rdf:type ?type .
}}
GROUP BY ?type
ORDER BY DESC(?count)
"""

entity_counts_df = run_query(
    "Entity counts by RDF class",
    entity_count_query,
)


ENTITY COUNTS BY RDF CLASS
Rows returned: 5


,type,count
0,http://www.opengis.net/ont/geosparql#Geometry,11683
1,https://example.org/smartcity/Building,10347
2,https://example.org/smartcity/Road,947
3,https://example.org/smartcity/POI,224
4,https://example.org/smartcity/Transport,165


### Query 5 — Service type distribution ###

In [9]:
service_count_query = f"""
PREFIX sc: <{SC}>

SELECT ?service (COUNT(?x) AS ?count)
WHERE {{
    ?x sc:serviceType ?service .
}}
GROUP BY ?service
ORDER BY DESC(?count)
"""

service_counts_df = run_query(
    "Service type distribution",
    service_count_query,
)


SERVICE TYPE DISTRIBUTION
Rows returned: 10


,service,count
0,bus_stop,164
1,park,20
2,school,12
3,supermarket,10
4,atm,6
5,pharmacy,5
6,bank,4
7,library,1
8,train_station,1
9,hospital,1


### Query 6 — Wikidata reconciliation count ###

In [10]:
wikidata_count_query = f"""
PREFIX owl: <{OWL_NS}>

SELECT (COUNT(?x) AS ?wikidata_links)
WHERE {{
    ?x owl:sameAs ?wd .
}}
"""

wikidata_count_df = run_query(
    "Wikidata reconciliation count",
    wikidata_count_query,
)


WIKIDATA RECONCILIATION COUNT
Rows returned: 1


,wikidata_links
0,5


### Query 7 — Entities with labels and geometry ###

In [11]:
geometry_query = f"""
PREFIX sc: <{SC}>
PREFIX rdfs: <{RDFS_NS}>

SELECT ?x ?label ?wkt
WHERE {{
    ?x rdfs:label ?label ;
       sc:wktGeometry ?wkt .
}}
LIMIT 100
"""

geometry_df = run_query(
    "Entities with labels and geometry",
    geometry_query,
)


ENTITIES WITH LABELS AND GEOMETRY
Rows returned: 0
No results found.


### Query 8 — Inspect all RDF predicates ###

In [12]:
predicate_query = f"""
SELECT ?predicate (COUNT(*) AS ?count)
WHERE {{
    ?s ?predicate ?o .
}}
GROUP BY ?predicate
ORDER BY DESC(?count)
"""

predicate_df = run_query(
    "RDF predicate inventory",
    predicate_query,
)


RDF PREDICATE INVENTORY
Rows returned: 7


,predicate,count
0,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,23366
1,http://www.opengis.net/ont/geosparql#asWKT,13091
2,http://www.opengis.net/ont/geosparql#hasGeometry,11683
3,http://www.w3.org/2000/01/rdf-schema#label,1396
4,https://example.org/smartcity/serviceType,224
5,https://example.org/smartcity/transportType,165
6,http://www.w3.org/2002/07/owl#sameAs,5


### Query 9 — Inspect Wikidata-linked entities with labels ###

In [13]:
wikidata_labels_query = f"""
PREFIX owl: <{OWL_NS}>
PREFIX rdfs: <{RDFS_NS}>

SELECT ?entity ?label ?wikidata
WHERE {{
    ?entity owl:sameAs ?wikidata ;
            rdfs:label ?label .
}}
ORDER BY ?label
LIMIT 100
"""

wikidata_labels_df = run_query(
    "Wikidata-linked entities",
    wikidata_labels_query,
)


WIKIDATA-LINKED ENTITIES
Rows returned: 5


,entity,label,wikidata
0,https://example.org/entity/poi_280734435,Alexander-von-Humboldt-Schule,https://www.wikidata.org/entity/Q116511061
1,https://example.org/entity/poi_179450968,Brüder-Grimm-Schule Eschwege,https://www.wikidata.org/entity/Q116511109
2,https://example.org/entity/poi_2315563270,Eschwege,https://www.wikidata.org/entity/Q800740
3,https://example.org/entity/poi_44806741,Friedrich-Wilhelm-Schule,https://www.wikidata.org/entity/Q44549210
4,https://example.org/entity/poi_12110384939,Rolf-Hochhuth-Stadtbibliothek,https://www.wikidata.org/entity/Q28682476


### Query 10 — Service entities with Wikidata links ###

In [14]:
service_wikidata_query = f"""
PREFIX sc: <{SC}>
PREFIX rdfs: <{RDFS_NS}>
PREFIX owl: <{OWL_NS}>

SELECT ?entity ?label ?service ?wikidata
WHERE {{
    ?entity sc:serviceType ?service ;
             rdfs:label ?label ;
             owl:sameAs ?wikidata .
}}
ORDER BY ?service ?label
LIMIT 100
"""

service_wikidata_df = run_query(
    "Services reconciled with Wikidata",
    service_wikidata_query,
)


SERVICES RECONCILED WITH WIKIDATA
Rows returned: 5


,entity,label,service,wikidata
0,https://example.org/entity/poi_12110384939,Rolf-Hochhuth-Stadtbibliothek,library,https://www.wikidata.org/entity/Q28682476
1,https://example.org/entity/poi_280734435,Alexander-von-Humboldt-Schule,school,https://www.wikidata.org/entity/Q116511061
2,https://example.org/entity/poi_179450968,Brüder-Grimm-Schule Eschwege,school,https://www.wikidata.org/entity/Q116511109
3,https://example.org/entity/poi_44806741,Friedrich-Wilhelm-Schule,school,https://www.wikidata.org/entity/Q44549210
4,https://example.org/entity/poi_2315563270,Eschwege,train_station,https://www.wikidata.org/entity/Q800740


In [15]:
# ============================================================
# FINAL SPARQL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("SPARQL ANALYSIS SUMMARY")
print("=" * 60)

print(f"""
Knowledge graph
---------------
RDF triples:              {len(g):,}

Query results
-------------
Services returned:        {len(services_df):,}
Buildings returned:       {len(buildings_df):,}
Wikidata links returned:  {len(wikidata_df):,}
Geometry entities:        {len(geometry_df):,}

Output
------
Source graph:
{GRAPH_PATH}
""")

print("✓ Notebook 05 completed successfully.")


SPARQL ANALYSIS SUMMARY

Knowledge graph
---------------
RDF triples:              49,930

Query results
-------------
Services returned:        100
Buildings returned:       100
Wikidata links returned:  5
Geometry entities:        0

Output
------
Source graph:
/Users/subhankarbiswas/smart-city-knowledge-graph-v3/outputs/graphs/smart_city_kg.ttl

✓ Notebook 05 completed successfully.
